# SOXX 2019 Mar/Jun/Sep 三份原始披露身份桥

SEC 申报日生效时钟保持原值；名称只作严格脚注规范化，现代 CUSIP 从原生 Qlib Recorder 读取。零市场结果。

In [1]:
import os,sys,json,hashlib,re,unicodedata,collections
from pathlib import Path
import qlib
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root));card=root/'research/stage1_soxx_basket_rotation_2019_three_report_bridge_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen'] and not (card/'run_state.json').exists();sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
mar=root/src['legacy_Mar2019'];jun=root/src['public_Jun2019'];modern=root/src['modern_card'];assert sha(mar)==src['legacy_sha256'] and sha(jun)==src['public_Jun_sha256'] and sha(modern/'summary.json')==src['modern_summary_sha256']
jun_summary=json.loads((jun.parent/'summary.json').read_text());assert jun_summary['source_NAS_sha256']==src['public_Jun_NAS_sha256'] and json.loads((jun.parent/'nas_source_archive.json').read_text())['remote_and_scoped_restore_verified']
modern_summary=json.loads((modern/'summary.json').read_text());artifact=modern/'qlib_recorder'/modern_summary['experiment_id']/modern_summary['recorder_id']/'artifacts/analysis/member_rows';assert sha(artifact)==src['modern_member_rows_sha256'] and modern_summary['nas_readback']['verified']
old_reports=json.loads(mar.read_text())['reports'];old=[x for x in old_reports if x['report_period']==plan['periods'][0]];assert len(old)==1 and len(old[0]['equities'])==30;old=old[0]
mid=json.loads(jun.read_text());assert mid['report_period']==plan['periods'][1] and len(mid['equities'])==30
state={'plan_sha256':sha(card/'plan.json'),'phase':'modern_Recorder_load_attempted','legacy_result_loads':1,'June_result_loads':1,'modern_Recorder_object_loads':1,'external_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/json.loads((modern/'plan.json').read_text())['calendar_provider']),region='us',kernels=1)
with R.uri_context(modern_summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=modern_summary['recorder_id'],experiment_id=modern_summary['experiment_id']);assert rec.status=='FINISHED';rows=rec.load_object('analysis/member_rows')
assert len(rows)==390;modern_periods=sorted({r['report_period'] for r in rows});assert modern_periods[0]==plan['periods'][2];last=[r for r in rows if r['report_period']==modern_periods[0]];assert len(last)==30
counts={'parenthesized_terminal_removed':0,'bare_lowercase_terminal_removed':0}
def norm(name):
 s=unicodedata.normalize('NFKC',name).strip()
 while True:
  m=re.search(r'\s*\([a-z]\)\s*$',s)
  if not m:break
  s=s[:m.start()].rstrip();counts['parenthesized_terminal_removed']+=1
 m=re.search(r'\s+[a-z](?:,[a-z])*\s*$',s)
 if m:s=s[:m.start()].rstrip();counts['bare_lowercase_terminal_removed']+=1
 s=re.sub(r'[.,]',' ',s)
 return re.sub(r'\s+',' ',s).strip().casefold()
first_by={norm(r['security_original']):r for r in old['equities']};mid_by={norm(r['security_original']):r for r in mid['equities']};last_by={norm(r['security_original']):r for r in last};assert len(first_by)==len(mid_by)==len(last_by)==30
shared_1=set(first_by)&set(mid_by);shared_2=set(mid_by)&set(last_by);matched=[{'normalized_security':name,'Jun_original_name':mid_by[name]['security_original'],'Sep_NPORT_name':last_by[name]['security_original'],'Sep_CUSIP':last_by[name]['cusip_original'],'Sep_identifier_status':last_by[name]['identifier_status']} for name in sorted(shared_2)]
valid=[x for x in matched if x['Sep_identifier_status']=='original_unique_CUSIP'];placeholder=[x for x in matched if x['Sep_identifier_status']!='original_unique_CUSIP'];assert len({x['Sep_CUSIP'] for x in valid})==len(valid)
result={'decision':'2019_three_public_SOXX_baskets_source_identity_bridge_partial','plan_sha256':sha(card/'plan.json'),'periods':plan['periods'],'accessions':[old['accession'],mid['accession'],last[0]['accession']],'Mar_to_Jun_exact_name_overlap':len(shared_1),'Jun_to_Sep_exact_name_overlap':len(shared_2),'Jun_to_Sep_matched_unique_original_CUSIP':len(valid),'Jun_to_Sep_matched_placeholder_CUSIP':len(placeholder),'Mar_only_names':sorted(set(first_by)-set(mid_by)),'Jun_new_vs_Mar_names':sorted(set(mid_by)-set(first_by)),'Jun_only_vs_Sep_names':sorted(set(mid_by)-set(last_by)),'Sep_new_vs_Jun_names':sorted(set(last_by)-set(mid_by)),'Jun_to_Sep_matched_identity_rows':matched,'modern_original_CUSIP_rows':sum(r['identifier_status']=='original_unique_CUSIP' for r in rows),'modern_placeholder_CUSIP_rows':sum(r['identifier_status']!='original_unique_CUSIP' for r in rows),'normalization_counts':counts,'Jun_first_conservative_Qlib_signal_date':mid['first_conservative_Qlib_signal_date'],'full_2011_2026_legal_security_identity_qualified':False,'turnover_feature_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_Recorder_id':modern_summary['recorder_id'],'source_NAS_sha256':modern_summary['nas_evidence']['sha256'],'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Exact names and CUSIPs support only a partial legal-identity bridge; unmatched and placeholder rows remain unknown.','A reported basket change is not fund executed turnover or a predictive SOXX factor.','No Stage1 0bp account or independent forward evidence.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({k:result[k] for k in ('periods','Mar_to_Jun_exact_name_overlap','Jun_to_Sep_exact_name_overlap','Jun_to_Sep_matched_unique_original_CUSIP','Jun_to_Sep_matched_placeholder_CUSIP','Mar_only_names','Jun_new_vs_Mar_names','Jun_only_vs_Sep_names','Sep_new_vs_Jun_names')})

[51855:MainThread](2026-10-23 10:04:10,263) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[51855:MainThread](2026-10-23 10:04:10,265) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[51855:MainThread](2026-10-23 10:04:10,265) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'periods': ['2019-03-31', '2019-06-30', '2019-09-30'], 'Mar_to_Jun_exact_name_overlap': 29, 'Jun_to_Sep_exact_name_overlap': 24, 'Jun_to_Sep_matched_unique_original_CUSIP': 22, 'Jun_to_Sep_matched_placeholder_CUSIP': 2, 'Mar_only_names': ['mellanox technologies ltd'], 'Jun_new_vs_Mar_names': ['inphi corp'], 'Jun_only_vs_Sep_names': ['asml holding nv nys', 'cypress semiconductor corp', 'inphi corp', 'kla-tencor corp', 'silicon motion technology corp adr', 'taiwan semiconductor manufacturing co ltd adr'], 'Sep_new_vs_Jun_names': ['asml holding nv', 'cabot microelectronics corp', 'cirrus logic inc', 'kla corp', 'semtech corp', 'tsmc']}


# SOXX 2019 年三份公开篮子：来源期连续、身份部分接桥

只读2019-03旧表单、2019-06公开 NPORT-EX Part F 和2019-09首份公开 NPORT-P 的原始 SOXX 30股向量。各报告依原 SEC 申报后的 Qlib 可用时钟使用，不按报告期提前回填；2019-06文件首个保守信号日为2019-08-27。现代成员对象从已完成且 NAS 回读过的原生 Qlib Recorder 读取。零新 HTTP、零 SOXX 未来收益、零拟合、零账户。

脚注/点号的固定来源规范化下，**2019-03→06 保留29/30**：Mellanox Technologies Ltd. 不再出现、Inphi Corp. 出现。**2019-06→09 保留24/30**，其中22项有原始唯一 CUSIP，Marvell 与 NXP 两项仍用 `000000000` 占位。六月与九月各有六个未匹配名；ASML、KLA、TSMC 属需核对的名称写法过渡，其他差异也不能仅凭字符串断定真实进出。现代13份 N-PORT 共390行，357行有唯一原始 CUSIP、33行是占位。完整名称与身份列表在 `result.json`。

**来源期断点已补，完整证券身份门仍未通过。** 名称交集只能提示披露篮子变化，披露市值随行情漂移，不等于基金实际交易换手；2011–2018早期别名、2019过渡写法及现代占位 CUSIP 需要独立原件核对后才能冻结无歧义的全期轮换字段。本卡未对 Stage1 或 SOXX 费用后表现作任何判断。
